<a href="https://colab.research.google.com/github/Binilts03/oaqjp-final-project-emb-ai/blob/main/nb/Gemma4_(E4B)-Audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [2]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [3]:
from unsloth import FastModel
import torch
from huggingface_hub import snapshot_download

fourbit_models = [
    # Gemma 4 models
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B-it",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth

model, processor = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None, # None for auto detection
    max_seq_length = 8192, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

# Fine-tuning Gemma 4 as a Regional English Tutor

This project focuses on fine-tuning **Gemma-4-E2B** to help Indian speakers (Hindi, Tamil, Telugu, Malayalam, Kannada) improve their English by removing Mother Tongue Influence (MTI).

In [4]:
from transformers import TextStreamer
# Helper function for inference
def do_gemma_4_inference(messages, max_new_tokens = 128):
    _ = model.generate(
        **processor.apply_chat_template(
            messages,
            add_generation_prompt = True, # Must add for generation
            tokenize = True,
            return_dict = True,
            return_tensors = "pt",
        ).to("cuda"),
        max_new_tokens = max_new_tokens,
        do_sample = False,
        streamer = TextStreamer(processor, skip_prompt = True),
    )

### Regional Language Data Integration
We combine datasets from **ai4bharat** to capture regional syntax and spoken nuances.

In [3]:
from datasets import load_dataset, Audio, concatenate_datasets, Dataset
from google.colab import userdata
from huggingface_hub import login, notebook_login, get_token
import os

# Authentication logic
try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        print("Successfully authenticated with HF_TOKEN from secrets.")
    else:
        raise ValueError()
except Exception:
    print('HF_TOKEN secret not found or invalid. Please login manually:')
    notebook_login()
    hf_token = get_token()

# Using the new dataset source provided: IndicVoices-ST
regional_languages = ['hindi', 'tamil', 'telugu', 'malayalam', 'kannada']
subsets = []

print('Loading IndicVoices-ST dataset subsets...')
for lang in regional_languages:
    try:
        print(f'Loading split: {lang}...')
        # Loading 2000 samples per language split from IndicVoices-ST
        lang_data = load_dataset('ai4bharat/IndicVoices-ST', 'indic2en', split=lang, streaming=True, token=hf_token).take(2000)
        lang_list = list(lang_data)
        if lang_list:
            subsets.append(Dataset.from_list(lang_list))
            print(f'Successfully loaded {len(lang_list)} samples for {lang}')
    except Exception as e:
        print(f'Error loading {lang}: {e}')

if not subsets:
    print('\nCRITICAL: No datasets loaded. Please check if you have accepted terms for ai4bharat/IndicVoices-ST on Hugging Face.')
else:
    dataset = concatenate_datasets(subsets).shuffle(seed=3407)

    def standardize_columns(example):
        return {
            'input': example.get('transcription', ''),
            'output': example.get('english_translation', ''),
            'instruction': f'The user spoke in {example.get("language", "a regional language")}. Translate to natural English and correct any Mother Tongue Influence.'
        }

    dataset = dataset.map(standardize_columns)
    print(f'Total training samples: {len(dataset)}')

Successfully authenticated with HF_TOKEN from secrets.
Loading IndicVoices-ST dataset subsets...
Loading split: hindi...
Successfully loaded 2000 samples for hindi
Loading split: tamil...
Successfully loaded 2000 samples for tamil
Loading split: telugu...
Successfully loaded 2000 samples for telugu
Loading split: malayalam...
Successfully loaded 2000 samples for malayalam
Loading split: kannada...
Successfully loaded 2000 samples for kannada


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Total training samples: 10000


In [ ]:
import requests
from datasets import load_dataset, Dataset, concatenate_datasets
from google.colab import userdata

# 1. Setup Auth
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

# 2. Re-verify IndicVoices-ST subsets (from previous step)
regional_languages = ['hindi', 'tamil', 'telugu', 'malayalam', 'kannada']
iv_subsets = []
print('Reloading IndicVoices-ST subsets...')
for lang in regional_languages:
    try:
        iv_data = load_dataset('ai4bharat/IndicVoices-ST', 'indic2en', split=lang, streaming=True, token=hf_token).take(2000)
        iv_list = list(iv_data)
        if iv_list: iv_subsets.append(Dataset.from_list(iv_list))
    except Exception as e: print(f'Error re-loading {lang}: {e}')

# 3. Load Spoken-Tutorial
st_subsets = []
dataset_name = 'ai4bharat/Spoken-Tutorial'
headers = {"Authorization": f"Bearer {hf_token}"} if hf_token else {}
splits_url = f"https://datasets-server.huggingface.co/splits?dataset={dataset_name.replace('/', '%2F')}"

try:
    response = requests.get(splits_url, headers=headers)
    response.raise_for_status()
    st_splits = [s['split'] for s in response.json()['splits'] if s['config'] == 'indic2en']
    print(f"Found Spoken-Tutorial splits: {st_splits}")
    for lang in st_splits:
        st_data = load_dataset(dataset_name, 'indic2en', split=lang, streaming=True, token=hf_token).take(1000)
        st_list = list(st_data)
        if st_list: st_subsets.append(Dataset.from_list(st_list))
except Exception as e: print(f"Error with Spoken-Tutorial: {e}")

# 4. Merge and Map
all_data = iv_subsets + st_subsets
dataset = concatenate_datasets(all_data).shuffle(seed=3407)

def standardize_combined(example):
    # Mapping logic to handle various field names in the source datasets
    input_text = example.get('text') or example.get('transcription') or ""
    output_text = example.get('en_text') or example.get('translation') or ""
    lang_name = example.get('language') or "regional language"
    return {
        'input': input_text,
        'output': output_text,
        'instruction': f'The user spoke in {lang_name}. Translate to natural English and correct any Mother Tongue Influence.'
    }

dataset = dataset.map(standardize_combined)
print(f'Final combined dataset size: {len(dataset)}')

Reloading IndicVoices-ST subsets...
Found Spoken-Tutorial splits: ['assamese', 'bengali', 'gujarati', 'hindi', 'kannada', 'malayalam', 'marathi', 'odia', 'punjabi', 'sanskrit', 'nepali', 'tamil', 'telugu']


### Training Progress
The model is being trained to recognize regional linguistic patterns and suggest natural English alternatives.

# Let's finetune Gemma 4!

You can finetune the vision and text and audio parts

We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
from unsloth import FastModel
import torch

# Ensure model is loaded
if 'model' not in locals():
    model, processor = FastModel.from_pretrained(
        model_name = "unsloth/gemma-4-E2B-it",
        dtype = None,
        load_in_4bit = True,
    )

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 8,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
        "post", "linear_start", "linear_end",
        "embedding_projection",
        "ffw_layer_1", "ffw_layer_2",
        "output_proj",
    ]
)
print("LoRA adapters successfully attached to Gemma-4.")

==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

### Data Preparation
We format the data into a conversational structure where the system acts as an expert English teacher. The user provides regional text (and audio where available), and the assistant provides corrected natural English.

In [27]:
def format_bhasa_data(samples: dict) -> dict[str, list]:
    formatted_samples = {"messages": []}
    keys = list(samples.keys())
    num_samples = len(samples[keys[0]])

    instructions = samples.get("instruction", ["Help me speak natural English"] * num_samples)
    inputs = samples.get("input", [""] * num_samples)
    outputs = samples.get("output", [""] * num_samples)

    for i in range(num_samples):
        # We prompt the model to be a specialized Indian English Teacher
        message = [
            {
                "role": "system",
                "content": [{"type": "text", "text": "You are an AI English Tutor for Indians. You understand the syntax of Hindi, Tamil, Telugu, Malayalam, and Kannada. Your goal is to help users speak English without mother tongue influence (MTI)."}]
            },
            {
                "role": "user",
                "content": [{"type": "text", "text": f"{instructions[i]}\nRegional Text: {inputs[i]}"}]
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": f"Natural English: {outputs[i]}"}]
            }
        ]

        # Insert audio if available in the dataset
        if "audio" in samples and samples["audio"][i] is not None:
            message[1]["content"].insert(0, {"type": "audio", "audio": samples["audio"][i]["array"]})

        formatted_samples["messages"].append(message)
    return formatted_samples

In [ ]:
import gc

# Reduce num_proc to 1 to save CPU RAM during mapping
dataset = dataset.map(format_bhasa_data, batched = True, batch_size = 4, num_proc = 1)

# Force garbage collection to free up system RAM before training
gc.collect()
torch.cuda.empty_cache()

print(f"Dataset fully prepared for training. Total samples: {len(dataset)}")

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [2]:
from unsloth import FastModel
import torch
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

# Ensure model, processor, and dataset are available
if 'model' not in locals():
    model, processor = FastModel.from_pretrained(
        model_name = "unsloth/gemma-4-E2B-it",
        dtype = None,
        load_in_4bit = True,
    )

if 'dataset' not in locals():
    raise NameError("The 'dataset' variable is missing. Please run the data preparation cells first.")

# Initializing the trainer
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    processing_class = processor.tokenizer,
    data_collator = UnslothVisionDataCollator(model, processor),
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 2,
        max_steps = -1,
        learning_rate = 2e-5,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        remove_unused_columns = False,
        dataset_text_field = "messages", # Point to the conversational column
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    )
)

NameError: The 'dataset' variable is missing. Please run the data preparation cells first.

In [ ]:
# Start the actual fine-tuning process
trainer_stats = trainer.train()

print("Fine-tuning complete!")

In [12]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
10.396 GB of memory reserved.


# Let's train the model!

To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [13]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 23,934,976 of 8,020,091,424 (0.30% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,15.100435
2,15.028904
3,14.713876
4,15.237208
5,14.527750
6,13.354699
7,12.372626
8,11.371757
9,10.759574
10,9.019562


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [14]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

212.2772 seconds used for training.
3.54 minutes used for training.
Peak reserved memory = 11.9 GB.
Peak reserved memory for training = 1.504 GB.
Peak reserved memory % of max memory = 81.714 %.
Peak reserved memory for training % of max memory = 10.328 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-4` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64` but for this example we use `do_sample=False` for ASR.

In [15]:
messages = [
    {
        "role": "system",
        "content": [
            {
                "type": "text",
                "text": "You are an assistant that transcribes speech accurately.",
            }
        ],
    },
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": test_audio['audio']['array']},
            {"type": "text", "text": "Please transcribe this audio."}
        ]
    }
]

do_gemma_4_inference(messages, max_new_tokens = 256)

Sie reden direkt mich an und muss mal finish klar, dass es politische Interessen gibt im Handel, im Austausch mit Waren, dass es politische Einflüsse gibt. Wie gerade ist die Alternative soll es nicht sein.<turn|>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [16]:
model.save_pretrained("gemma_4_lora")  # Local saving
processor.save_pretrained("gemma_4_lora")
# model.push_to_hub("HF_ACCOUNT/gemma_4_lora", token = "YOUR_HF_TOKEN") # Online saving
# processor.push_to_hub("HF_ACCOUNT/gemma_4_lora", token = "YOUR_HF_TOKEN") # Online saving

Unsloth: Restored added_tokens_decoder metadata in gemma_4_lora/tokenizer_config.json.


['gemma_4_lora/processor_config.json']

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [17]:
if False:
    from unsloth import FastModel
    model, processor = FastModel.from_pretrained(
        model_name = "gemma_4_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "What is Gemma-4?",}]
}]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 128, # Increase for longer outputs!
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(processor, skip_prompt = True),
)

I am Gemma 4. I am a Large Language Model developed by Google DeepMind. I am an open weights model.<turn|>


### Saving to float16 for VLLM

We also support saving to `float16` directly for deployment! We save it in the folder `gemma-4-finetune`. Set `if False` to `if True` to let it run!

In [18]:
if False: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-4", processor)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [19]:
if False: # Change to True to upload finetune
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-4-finetune", processor,
        token = "YOUR_HF_TOKEN"
    )

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now for all models! For now, you can convert easily to `Q8_0, F16 or BF16` precision. `Q4_K_M` for 4bit will come later!

In [20]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "gemma_4_finetune",
        processor,
        quantization_method = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [21]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "HF_ACCOUNT/gemma_4_finetune",
        processor,
        quantization_method = "Q8_0", # Only Q8_0, BF16, F16 supported
        token = "YOUR_HF_TOKEN",
    )

Now, use the `gemma-4-finetune.gguf` file or `gemma-4-finetune-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).